In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
from glob import glob

In [9]:
import pandas as pd

# Dataset paths
datasets = {
    "fund_master": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/01_fund_master.csv",

    "nav_history": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/02_nav_history.csv",

    "aum_by_fund_house": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/03_aum_by_fund_house.csv",

    "monthly_sip_inflows": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/04_monthly_sip_inflows.csv",

    "category_inflows": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/05_category_inflows.csv",

    "industry_folio_count": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/06_industry_folio_count.csv",

    "scheme_performance": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/07_scheme_performance.csv",

    "investor_transactions": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/08_investor_transactions.csv",

    "portfolio_holdings": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/09_portfolio_holdings.csv",

    "benchmark_indices": "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/10_benchmark_indices.csv"
}

# Dictionary to store loaded dataframes
loaded_data = {}

# Load and inspect all datasets
for name, path in datasets.items():

    print("\n" + "="*100)
    print(f"DATASET: {name.upper()}")
    print("="*100)

    try:
        # Load dataset
        df = pd.read_csv(path)

        # Store dataframe
        loaded_data[name] = df

        # Shape
        print("\n1. Shape:")
        print(df.shape)

        # Data types
        print("\n2. Data Types:")
        print(df.dtypes)

        # First 5 rows
        print("\n3. First 5 Rows:")
        print(df.head())

        # Missing values
        print("\n4. Missing Values:")
        print(df.isnull().sum())

        # Duplicate rows
        print("\n5. Duplicate Rows:")
        print(df.duplicated().sum())

    except Exception as e:
        print(f"\nError loading {name}: {e}")

print("\n" + "="*100)
print("ALL DATASETS LOADED SUCCESSFULLY")
print("="*100)


DATASET: FUND_MASTER

1. Shape:
(40, 15)

2. Data Types:
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object

3. First 5 Rows:
   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular Pla

All 10 datasets loaded successfully.
Some datasets contain missing values.
Duplicate rows detected in few datasets.
Date columns require datatype conversion.
Some numerical columns are stored as object datatype.
Data cleaning and preprocessing will be required before analysis.

In [10]:
import requests
import pandas as pd

# API URL
url = "https://api.mfapi.in/mf/125497"

# Request API
response = requests.get(url)

# Convert JSON
data = response.json()

# Print Scheme Name
print("Scheme Name:", data['meta']['scheme_name'])

# Convert NAV data into DataFrame
nav_df = pd.DataFrame(data['data'])

# Display first rows
print(nav_df.head())

# Save CSV
nav_df.to_csv(
    "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/hdfc_top100_live_nav.csv",
    index=False
)

print("Live NAV CSV saved successfully")

Scheme Name: SBI Small Cap Fund - Direct Plan - Growth
         date        nav
0  01-06-2026  192.31950
1  31-05-2026  193.68360
2  29-05-2026  193.68480
3  27-05-2026  195.05010
4  26-05-2026  194.22580
Live NAV CSV saved successfully


In [11]:
import requests
import pandas as pd
import time

schemes = {
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_Large_Cap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

all_nav_data = []

for scheme_name, scheme_code in schemes.items():

    print(f"Fetching {scheme_name}")

    url = f"https://api.mfapi.in/mf/{scheme_code}"

    response = requests.get(url)

    data = response.json()

    temp_df = pd.DataFrame(data['data'])

    temp_df['scheme_name'] = scheme_name
    temp_df['scheme_code'] = scheme_code

    all_nav_data.append(temp_df)

    time.sleep(1)

# Combine all datasets
final_nav_df = pd.concat(all_nav_data, ignore_index=True)

# Save CSV
final_nav_df.to_csv(
    "/Users/chitranjansahu/Desktop/mutual_fund_analysis/data/raw/key_schemes_nav.csv",
    index=False
)

print(final_nav_df.head())
print("All NAV data saved successfully")

Fetching SBI_Bluechip
Fetching ICICI_Bluechip
Fetching Nippon_Large_Cap
Fetching Axis_Bluechip
Fetching Kotak_Bluechip
         date        nav   scheme_name  scheme_code
0  01-06-2026  104.70250  SBI_Bluechip       119551
1  29-05-2026  104.65700  SBI_Bluechip       119551
2  27-05-2026  104.58330  SBI_Bluechip       119551
3  26-05-2026  104.52690  SBI_Bluechip       119551
4  25-05-2026  104.52150  SBI_Bluechip       119551
All NAV data saved successfully


In [18]:
fund_master = loaded_data['fund_master']

print("Unique Fund Houses")
print(fund_master['fund_house'].unique())

print("\nCategories")
print(fund_master['category'].unique())

print("\nSub Categories")
print(fund_master['sub_category'].unique())

print("\nRisk Grades")
print(fund_master['risk_category'].unique())

Unique Fund Houses
['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']

Categories
['Equity' 'Debt']

Sub Categories
['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']

Risk Grades
['Moderate' 'Very High' 'Low' 'High' 'Moderately High']


In [19]:
fund_master.columns


Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='object')

In [20]:
fund_master = loaded_data['fund_master']
nav_history = loaded_data['nav_history']

# Convert to string
master_codes = set(fund_master['amfi_code'].astype(str))
nav_codes = set(nav_history['amfi_code'].astype(str))

# Find missing codes
missing_codes = master_codes - nav_codes

print("Total Fund Master Codes:", len(master_codes))
print("Total NAV History Codes:", len(nav_codes))
print("Missing Codes:", len(missing_codes))

print("\nSample Missing Codes:")
print(list(missing_codes)[:10])

Total Fund Master Codes: 40
Total NAV History Codes: 40
Missing Codes: 0

Sample Missing Codes:
[]


All datasets loaded successfully.
Missing values found in multiple datasets.
Duplicate rows identified in some datasets.
Date columns need conversion to datetime datatype.
Some numerical columns stored as object datatype.
Few AMFI codes are missing in NAV history.
Data preprocessing and cleaning required before analysis.